# GSM2008: Global Sky Model

Based on de Oliveira-Costa et al. (2008), MNRAS 388, 247.  
PCA-based all-sky model of diffuse Galactic radio emission (10 MHz – 100 GHz).  
3 principal components fitted to 11 total-power radio surveys, capturing 99.7% of variance.

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pygdsm import GlobalSkyModel

from rrivis.core.sky import SkyModel

%matplotlib inline
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100})

### List available models

In [ ]:
# Show all available diffuse models and their descriptions
for name, desc in SkyModel.list_diffuse_models().items():
    print(f"\n{'=' * 80}")
    print(f"  {name.upper()}")
    print(f"{'=' * 80}")
    print(desc)

### Generate GSM2008 maps at key frequencies

We generate maps at the 11 survey frequencies used in the original PCA fit, plus a few EoR-relevant frequencies.

In [ ]:
# The 11 original survey frequencies (Hz) used in the GSM2008 PCA
survey_freqs_hz = np.array([
    10e6, 22e6, 45e6, 408e6,         # sub-GHz radio
    1420e6, 2326e6,                    # L/S band
    23e9, 33e9, 41e9, 61e9, 94e9,     # WMAP bands
])

# EoR-relevant frequencies (50-250 MHz)
eor_freqs_hz = np.linspace(50e6, 250e6, 21)

# Generate using pygdsm directly — native resolution is nside=512
gsm = GlobalSkyModel(freq_unit="Hz", basemap="haslam", interpolation="pchip")

nside = 512  # native GSM2008 resolution

# Generate maps at survey frequencies
survey_maps = {}
for freq in survey_freqs_hz:
    m = gsm.generate(freq)
    survey_maps[freq] = hp.ud_grade(m, nside_out=nside)

# Generate maps at EoR frequencies
eor_maps = {}
for freq in eor_freqs_hz:
    m = gsm.generate(freq)
    eor_maps[freq] = hp.ud_grade(m, nside_out=nside)

print(f"Generated {len(survey_maps)} survey-frequency maps and {len(eor_maps)} EoR maps")
print(f"Map resolution: nside={nside}, npix={hp.nside2npix(nside)}, "
      f"pixel size={hp.nside2resol(nside, arcmin=True):.1f} arcmin")

### Mollweide projections at survey frequencies

All-sky maps in Galactic coordinates at each of the 11 frequencies used in the original PCA fit.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 20), subplot_kw={"projection": "mollweide"})
axes_flat = axes.flatten()

for i, (freq, m) in enumerate(survey_maps.items()):
    ax = axes_flat[i]
    # Convert HEALPix map to Mollweide image
    theta, phi = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)))
    # healpy mollview is easier here
    plt.sca(ax)
    if freq < 1e9:
        label = f"{freq/1e6:.0f} MHz"
    else:
        label = f"{freq/1e9:.0f} GHz"
    hp.mollview(np.log10(np.clip(m, 1e-6, None)), title=f"GSM2008 @ {label}  (log$_{{10}}$ T [K])",
                hold=True, cmap="inferno", notext=True)

# Hide last empty subplot
axes_flat[-1].set_visible(False)

plt.tight_layout()
plt.show()

### Global statistics at each survey frequency

Temperature statistics across the full sky at each of the 11 PCA input frequencies.

In [ ]:
print(f"{'Frequency':>12s}  {'Min':>12s}  {'Max':>12s}  {'Mean':>12s}  {'Median':>12s}  "
      f"{'Std':>12s}  {'Skewness':>10s}  {'Kurtosis':>10s}")
print("-" * 110)

from scipy import stats as spstats

for freq, m in survey_maps.items():
    if freq < 1e9:
        label = f"{freq/1e6:.0f} MHz"
    else:
        label = f"{freq/1e9:.0f} GHz"

    sk = spstats.skew(m)
    ku = spstats.kurtosis(m)

    # Format values with appropriate units
    def fmt(v):
        if abs(v) >= 1:
            return f"{v:.2f} K"
        elif abs(v) >= 1e-3:
            return f"{v*1e3:.2f} mK"
        else:
            return f"{v*1e6:.2f} uK"

    print(f"{label:>12s}  {fmt(m.min()):>12s}  {fmt(m.max()):>12s}  {fmt(m.mean()):>12s}  "
          f"{fmt(np.median(m)):>12s}  {fmt(m.std()):>12s}  {sk:>10.2f}  {ku:>10.2f}")

### Mean sky temperature vs frequency (global spectrum)

The overall spectral shape of diffuse Galactic emission — dominated by synchrotron at low frequencies, with dust rising at high frequencies.

In [ ]:
# Generate a dense frequency grid for the global spectrum
# pygdsm GSM2008 valid range: 10 MHz to 94 GHz (strict upper bound)
freq_grid = np.geomspace(10e6, 93.9e9, 200)

t_mean = np.zeros_like(freq_grid)
t_median = np.zeros_like(freq_grid)
t_min = np.zeros_like(freq_grid)
t_max = np.zeros_like(freq_grid)
t_p10 = np.zeros_like(freq_grid)  # 10th percentile (cleanest sky)
t_p90 = np.zeros_like(freq_grid)  # 90th percentile (dirtiest sky)

for i, freq in enumerate(freq_grid):
    m = gsm.generate(freq)
    m = hp.ud_grade(m, nside_out=64)  # lower res for speed
    t_mean[i] = m.mean()
    t_median[i] = np.median(m)
    t_min[i] = m.min()
    t_max[i] = m.max()
    t_p10[i] = np.percentile(m, 10)
    t_p90[i] = np.percentile(m, 90)

print("Done generating global spectrum.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})

freq_mhz = freq_grid / 1e6

# Top: absolute temperature
ax1.fill_between(freq_mhz, t_min, t_max, alpha=0.15, color="C0", label="Min–Max range")
ax1.fill_between(freq_mhz, t_p10, t_p90, alpha=0.3, color="C0", label="10th–90th percentile")
ax1.loglog(freq_mhz, t_mean, "C0-", lw=2, label="Mean")
ax1.loglog(freq_mhz, t_median, "C1--", lw=1.5, label="Median")

# Mark the 11 survey frequencies
survey_mhz = survey_freqs_hz / 1e6
survey_means = [survey_maps[f].mean() for f in survey_freqs_hz]
ax1.scatter(survey_mhz, survey_means, c="red", s=60, zorder=5, label="11 PCA input surveys")

# Reference power law: T ~ nu^-2.5
ref_freq = 150  # MHz
ref_temp = np.interp(ref_freq, freq_mhz, t_mean)
powerlaw = ref_temp * (freq_mhz / ref_freq) ** (-2.5)
ax1.loglog(freq_mhz, powerlaw, "k:", alpha=0.5, label=r"$\nu^{-2.5}$ reference")

ax1.set_ylabel("Brightness Temperature [K]")
ax1.set_title("GSM2008 Global Sky Spectrum")
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, which="both", alpha=0.3)

# Mark EoR band
ax1.axvspan(50, 250, alpha=0.08, color="green")
ax1.text(120, ax1.get_ylim()[1] * 0.5, "EoR band", fontsize=10, color="green",
         ha="center", va="top", alpha=0.7)

# Bottom: spectral index beta = d(log T) / d(log nu)
dlnT = np.diff(np.log(t_mean))
dlnnu = np.diff(np.log(freq_grid))
beta = dlnT / dlnnu
beta_freq = np.sqrt(freq_grid[:-1] * freq_grid[1:])  # geometric mean

ax2.semilogx(beta_freq / 1e6, beta, "C0-", lw=1.5)
ax2.axhline(-2.5, color="k", ls=":", alpha=0.5, label=r"$\beta = -2.5$")
ax2.axhline(-2.0, color="gray", ls=":", alpha=0.3)
ax2.axvspan(50, 250, alpha=0.08, color="green")
ax2.set_xlabel("Frequency [MHz]")
ax2.set_ylabel(r"Spectral index $\beta$")
ax2.set_ylim(-3.5, 0.5)
ax2.legend(fontsize=9)
ax2.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

---

## Basemap comparison: haslam vs wmap vs 5deg

The GSM2008 PCA was performed at 5.1-degree resolution. Higher-resolution maps are obtained by
"locking" the amplitude to a higher-resolution input map while keeping the PCA spectral shape:
- **haslam** (1 deg) — locked to 408 MHz Haslam map, recommended below 1 GHz
- **wmap** (2 deg) — locked to WMAP 23 GHz, recommended at CMB frequencies
- **5deg** (5.1 deg) — native PCA resolution, uses all 11 maps equally

In [ ]:
# Generate maps for all 3 basemaps at representative frequencies
basemaps = ["haslam", "wmap", "5deg"]
basemap_labels = {"haslam": "haslam (1 deg)", "wmap": "wmap (2 deg)", "5deg": "5deg (5.1 deg)"}

# Frequencies spanning the full range: low-freq (where haslam excels), mid, high (where wmap excels)
compare_freqs_mhz = [150, 408, 1420, 23000]
compare_labels = ["150 MHz", "408 MHz", "1.42 GHz", "23 GHz"]

nside_compare = 128  # lower nside for faster comparison

basemap_data = {}
for bm in basemaps:
    gsm_bm = GlobalSkyModel(freq_unit="MHz", basemap=bm, interpolation="pchip")
    basemap_data[bm] = {}
    for freq in compare_freqs_mhz:
        m = gsm_bm.generate(freq)
        basemap_data[bm][freq] = hp.ud_grade(m, nside_out=nside_compare)

print("Generated maps for all basemap x frequency combinations.")

In [ ]:
# Mollweide maps: rows = frequencies, columns = basemaps
fig, axes = plt.subplots(len(compare_freqs_mhz), len(basemaps), figsize=(18, 20))

for row, (freq, flabel) in enumerate(zip(compare_freqs_mhz, compare_labels)):
    for col, bm in enumerate(basemaps):
        m = basemap_data[bm][freq]
        plt.sca(axes[row, col])
        hp.mollview(
            np.log10(np.clip(m, 1e-6, None)),
            title=f"{flabel} — {basemap_labels[bm]}",
            hold=True, cmap="inferno", notext=True,
        )

plt.suptitle("GSM2008 basemap comparison (log$_{10}$ T [K])", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### Basemap fractional difference maps

Fractional difference relative to the 5deg (native PCA) basemap: `(T_basemap - T_5deg) / T_5deg`.
This shows where the higher-resolution amplitude locking introduces spatial structure beyond the PCA.

In [ ]:
# Fractional difference maps: haslam and wmap relative to 5deg
fig, axes = plt.subplots(len(compare_freqs_mhz), 2, figsize=(14, 20))

for row, (freq, flabel) in enumerate(zip(compare_freqs_mhz, compare_labels)):
    ref = basemap_data["5deg"][freq]
    for col, bm in enumerate(["haslam", "wmap"]):
        diff = basemap_data[bm][freq]
        frac = (diff - ref) / np.abs(ref)
        frac = np.clip(frac, -0.5, 0.5)  # clip for display
        plt.sca(axes[row, col])
        hp.mollview(
            frac,
            title=f"{flabel} — ({bm} - 5deg) / 5deg",
            hold=True, cmap="RdBu_r", min=-0.3, max=0.3, notext=True,
        )

plt.suptitle("Fractional difference from 5deg basemap", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mean sky spectrum comparison across basemaps
freq_grid_bm = np.geomspace(10, 93900, 150)  # MHz

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

colors = {"haslam": "C0", "wmap": "C1", "5deg": "C2"}

spectra = {}
for bm in basemaps:
    gsm_bm = GlobalSkyModel(freq_unit="MHz", basemap=bm, interpolation="pchip")
    t_mean_bm = np.zeros_like(freq_grid_bm)
    for i, freq in enumerate(freq_grid_bm):
        m = gsm_bm.generate(freq)
        m = hp.ud_grade(m, nside_out=64)
        t_mean_bm[i] = m.mean()
    spectra[bm] = t_mean_bm
    ax1.loglog(freq_grid_bm, t_mean_bm, color=colors[bm], lw=2,
               label=basemap_labels[bm])

ax1.set_ylabel("Mean T$_{\\rm sky}$ [K]")
ax1.set_title("Mean sky temperature by basemap")
ax1.legend(fontsize=10)
ax1.grid(True, which="both", alpha=0.3)

# Fractional difference relative to 5deg
for bm in ["haslam", "wmap"]:
    frac = (spectra[bm] - spectra["5deg"]) / spectra["5deg"]
    ax2.semilogx(freq_grid_bm, frac * 100, color=colors[bm], lw=1.5,
                 label=f"{bm} vs 5deg")

ax2.axhline(0, color="k", ls="-", lw=0.5)
ax2.set_xlabel("Frequency [MHz]")
ax2.set_ylabel("Fractional diff [%]")
ax2.set_ylim(-15, 15)
ax2.legend(fontsize=10)
ax2.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

---

## Interpolation comparison: pchip vs cubic

The 3 PCA component spectra and the scaling factor are defined at only the 11 survey frequencies.
Between these knots, pygdsm interpolates in log-frequency space using either:
- **pchip** (default) — piecewise cubic Hermite interpolating polynomial, monotone, guaranteed no overshoot
- **cubic** — standard cubic spline, smooth 1st/2nd derivatives, can overshoot between knots

Neither matches the original Fortran GSM code exactly (which used Numerical Recipes routines).

In [ ]:
# Compare pchip vs cubic interpolation — mean sky spectrum
interp_methods = ["pchip", "cubic"]
interp_labels = {"pchip": "PCHIP (default)", "cubic": "Cubic spline"}
interp_colors = {"pchip": "C0", "cubic": "C3"}

freq_grid_interp = np.geomspace(10, 93900, 200)  # MHz

interp_spectra = {}
for method in interp_methods:
    gsm_interp = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation=method)
    t_arr = np.zeros_like(freq_grid_interp)
    for i, freq in enumerate(freq_grid_interp):
        m = gsm_interp.generate(freq)
        m = hp.ud_grade(m, nside_out=64)
        t_arr[i] = m.mean()
    interp_spectra[method] = t_arr

print("Done generating spectra for both interpolation methods.")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), gridspec_kw={"height_ratios": [2, 1, 1]})

# Top: overlaid mean spectra
ax1 = axes[0]
for method in interp_methods:
    ax1.loglog(freq_grid_interp, interp_spectra[method], color=interp_colors[method],
               lw=2, label=interp_labels[method])

# Mark the 11 PCA knot frequencies
knot_freqs_mhz = np.array([10, 22, 45, 408, 1420, 2326, 23000, 33000, 41000, 61000, 94000])
knot_temps = np.interp(knot_freqs_mhz, freq_grid_interp, interp_spectra["pchip"])
ax1.scatter(knot_freqs_mhz, knot_temps, c="k", s=40, zorder=5, marker="o", label="PCA knots")

ax1.set_ylabel("Mean T$_{\\rm sky}$ [K]")
ax1.set_title("Interpolation method comparison (haslam basemap)")
ax1.legend(fontsize=10)
ax1.grid(True, which="both", alpha=0.3)

# Middle: fractional difference
ax2 = axes[1]
frac_diff = (interp_spectra["cubic"] - interp_spectra["pchip"]) / interp_spectra["pchip"]
ax2.semilogx(freq_grid_interp, frac_diff * 100, "C3-", lw=1.5)
ax2.axhline(0, color="k", ls="-", lw=0.5)
for kf in knot_freqs_mhz:
    ax2.axvline(kf, color="gray", ls=":", alpha=0.3)
ax2.set_ylabel("(cubic - pchip) / pchip [%]")
ax2.set_ylim(-10, 10)
ax2.grid(True, which="both", alpha=0.3)

# Bottom: spectral index comparison
ax3 = axes[2]
for method in interp_methods:
    dlnT = np.diff(np.log(np.clip(interp_spectra[method], 1e-10, None)))
    dlnnu = np.diff(np.log(freq_grid_interp))
    beta = dlnT / dlnnu
    beta_freq = np.sqrt(freq_grid_interp[:-1] * freq_grid_interp[1:])
    ax3.semilogx(beta_freq, beta, color=interp_colors[method], lw=1.5,
                 label=interp_labels[method])

ax3.axhline(-2.5, color="k", ls=":", alpha=0.5, label=r"$\beta = -2.5$")
for kf in knot_freqs_mhz:
    ax3.axvline(kf, color="gray", ls=":", alpha=0.3)
ax3.set_xlabel("Frequency [MHz]")
ax3.set_ylabel(r"Spectral index $\beta$")
ax3.set_ylim(-3.5, 1.0)
ax3.legend(fontsize=9)
ax3.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

### Interpolation difference maps at selected frequencies

Mollweide maps of (cubic - pchip) / pchip to show how interpolation differences vary spatially.
Differences are largest between PCA knot frequencies where the interpolants diverge.

In [ ]:
# Spatial difference maps: pchip vs cubic at frequencies between PCA knots
# These are frequencies where interpolation differences are typically largest
interp_diff_freqs = [15, 100, 800, 5000, 30000, 80000]  # MHz
interp_diff_labels = ["15 MHz", "100 MHz", "800 MHz", "5 GHz", "30 GHz", "80 GHz"]

gsm_pchip = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation="pchip")
gsm_cubic = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation="cubic")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (freq, flabel) in enumerate(zip(interp_diff_freqs, interp_diff_labels)):
    m_pchip = hp.ud_grade(gsm_pchip.generate(freq), nside_out=64)
    m_cubic = hp.ud_grade(gsm_cubic.generate(freq), nside_out=64)
    frac = (m_cubic - m_pchip) / np.abs(m_pchip)
    frac = np.clip(frac, -0.2, 0.2)

    row, col = divmod(i, 3)
    plt.sca(axes[row, col])
    hp.mollview(
        frac, title=f"{flabel} — (cubic - pchip) / pchip",
        hold=True, cmap="RdBu_r", min=-0.1, max=0.1, notext=True,
    )

plt.suptitle("Interpolation method spatial differences", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

## CMB inclusion comparison: with vs without the 2.725 K monopole

By default `include_cmb=False` and the GSM returns only Galactic foreground emission.
Setting `include_cmb=True` adds a uniform 2.725 K CMB monopole to every pixel.

At low frequencies Galactic emission dominates by orders of magnitude, so the CMB is negligible.
At high frequencies (tens of GHz) the foreground drops below the CMB in clean sky regions,
making the monopole a significant fraction of the total sky temperature.

In [ ]:
# Generate mean spectra with and without CMB
gsm_no_cmb = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation="pchip", include_cmb=False)
gsm_with_cmb = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation="pchip", include_cmb=True)

freq_grid_cmb = np.geomspace(10, 93900, 200)  # MHz
T_CMB = 2.725  # K

stats_no_cmb = {"mean": [], "median": [], "p10": [], "p90": [], "min": []}
stats_with_cmb = {"mean": [], "median": [], "p10": [], "p90": [], "min": []}

for freq in freq_grid_cmb:
    m0 = hp.ud_grade(gsm_no_cmb.generate(freq), nside_out=64)
    m1 = hp.ud_grade(gsm_with_cmb.generate(freq), nside_out=64)
    for m, d in [(m0, stats_no_cmb), (m1, stats_with_cmb)]:
        d["mean"].append(m.mean())
        d["median"].append(np.median(m))
        d["p10"].append(np.percentile(m, 10))
        d["p90"].append(np.percentile(m, 90))
        d["min"].append(m.min())

for d in [stats_no_cmb, stats_with_cmb]:
    for k in d:
        d[k] = np.array(d[k])

print("Done generating CMB comparison spectra.")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14), gridspec_kw={"height_ratios": [2, 1, 1]})

# Top: mean spectrum with/without CMB
ax1 = axes[0]
ax1.loglog(freq_grid_cmb, stats_no_cmb["mean"], "C0-", lw=2, label="Foreground only (mean)")
ax1.loglog(freq_grid_cmb, stats_with_cmb["mean"], "C1--", lw=2, label="Foreground + CMB (mean)")
ax1.fill_between(freq_grid_cmb, stats_no_cmb["p10"], stats_no_cmb["p90"],
                 alpha=0.15, color="C0", label="Foreground 10th–90th %ile")
ax1.axhline(T_CMB, color="red", ls=":", lw=1.5, label=f"T$_{{\\rm CMB}}$ = {T_CMB} K")
ax1.set_ylabel("Brightness Temperature [K]")
ax1.set_title("Effect of CMB monopole inclusion")
ax1.legend(fontsize=9, loc="upper right")
ax1.grid(True, which="both", alpha=0.3)

# Middle: CMB fraction of total sky temperature
ax2 = axes[1]
cmb_frac_mean = T_CMB / stats_with_cmb["mean"] * 100
cmb_frac_median = T_CMB / stats_with_cmb["median"] * 100
cmb_frac_p10 = T_CMB / stats_with_cmb["p10"] * 100  # coldest sky = highest CMB fraction

ax2.semilogx(freq_grid_cmb, cmb_frac_mean, "C1-", lw=2, label="CMB / (FG + CMB), mean sky")
ax2.semilogx(freq_grid_cmb, np.clip(cmb_frac_p10, 0, 100), "C3--", lw=1.5,
             label="CMB / (FG + CMB), 10th %ile (coldest)")
ax2.axhline(50, color="gray", ls=":", alpha=0.5)
ax2.set_ylabel("CMB fraction [%]")
ax2.set_ylim(0, 105)
ax2.legend(fontsize=9)
ax2.grid(True, which="both", alpha=0.3)

# Bottom: spectral index with and without CMB
ax3 = axes[2]
for data, label, color, ls in [
    (stats_no_cmb["mean"], "Foreground only", "C0", "-"),
    (stats_with_cmb["mean"], "Foreground + CMB", "C1", "--"),
]:
    dlnT = np.diff(np.log(np.clip(data, 1e-10, None)))
    dlnnu = np.diff(np.log(freq_grid_cmb))
    beta = dlnT / dlnnu
    beta_freq = np.sqrt(freq_grid_cmb[:-1] * freq_grid_cmb[1:])
    ax3.semilogx(beta_freq, beta, color=color, ls=ls, lw=1.5, label=label)

ax3.axhline(-2.5, color="k", ls=":", alpha=0.5, label=r"$\beta = -2.5$")
ax3.set_xlabel("Frequency [MHz]")
ax3.set_ylabel(r"Spectral index $\beta$")
ax3.set_ylim(-3.5, 1.0)
ax3.legend(fontsize=9)
ax3.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

### CMB difference maps at selected frequencies

Maps of `T_CMB / T_total` showing the CMB contribution fraction per pixel.
At low frequencies the CMB is negligible everywhere; at high frequencies it dominates in the
cleanest (high Galactic latitude) regions.

In [ ]:
# CMB fraction maps at selected frequencies
cmb_map_freqs = [150, 1420, 10000, 33000, 61000, 94000]  # MHz
cmb_map_labels = ["150 MHz", "1.42 GHz", "10 GHz", "33 GHz", "61 GHz", "94 GHz"]

gsm_cmb = GlobalSkyModel(freq_unit="MHz", basemap="haslam", interpolation="pchip", include_cmb=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (freq, flabel) in enumerate(zip(cmb_map_freqs, cmb_map_labels)):
    m_total = hp.ud_grade(gsm_cmb.generate(freq), nside_out=64)
    cmb_fraction = T_CMB / np.clip(m_total, 1e-10, None)
    cmb_fraction = np.clip(cmb_fraction, 0, 1)

    row, col = divmod(i, 3)
    plt.sca(axes[row, col])
    hp.mollview(
        cmb_fraction, title=f"{flabel} — T$_{{\\rm CMB}}$ / T$_{{\\rm total}}$",
        hold=True, cmap="magma", min=0, max=1, notext=True,
    )

plt.suptitle("CMB monopole fraction of total sky temperature", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()